In [21]:

using Plots, Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, JLD2,  Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, ForwardDiff, LinearAlgebra, Wandb

include("creation_model.jl")
# include("optimisation.jl")
include("optimisation_constant_inhibition.jl")


ERROR: LoadError: LoadError: syntax: invalid escape sequence
Stacktrace:
  [1] top-level scope
    @ ~/.julia/packages/Wandb/nfzlz/src/main.jl:38
  [2] include(mod::Module, _path::String)
    @ Base ./Base.jl:384
  [3] include(x::String)
    @ Wandb ~/.julia/packages/Wandb/nfzlz/src/Wandb.jl:1
  [4] top-level scope
    @ ~/.julia/packages/Wandb/nfzlz/src/Wandb.jl:33
  [5] include
    @ ./Base.jl:384 [inlined]
  [6] include_package_for_output(pkg::Base.PkgId, input::String, depot_path::Vector{String}, dl_load_path::Vector{String}, load_path::Vector{String}, concrete_deps::Vector{Pair{Base.PkgId, UInt64}}, source::String)
    @ Base ./loading.jl:1235
  [7] top-level scope
    @ none:1
  [8] eval
    @ ./boot.jl:360 [inlined]
  [9] eval(x::Expr)
    @ Base.MainInclude ./client.jl:446
 [10] top-level scope
    @ none:1
in expression starting at /Users/nmt121/.julia/packages/Wandb/nfzlz/src/main.jl:38
in expression starting at /Users/nmt121/.julia/packages/Wandb/nfzlz/src/Wandb.jl:1


ErrorException: Failed to precompile Wandb [ad70616a-06c9-5745-b1f1-6a5f42545108] to /Users/nmt121/.julia/compiled/v1.6/Wandb/jl_Ys2Lgj.

In [20]:
 Pkg.rm("Wandb"); Pkg.add("Wandb")


    Updating `~/.julia/environments/v1.6/Project.toml`
  [ad70616a] - Wandb v0.5.1
    Updating `~/.julia/environments/v1.6/Manifest.toml`
  [992eb4ea] - CondaPkg v0.2.22
  [0f8b85d8] - JSON3 v1.14.0
  [0b3b1443] - MicroMamba v0.1.14
  [fa939f87] - Pidfile v1.3.0
  [6099a3de] - PythonCall v0.9.19
  [856f2bd8] - StructTypes v1.10.0
  [e17b2a0c] - UnsafePointers v1.0.0
  [ad70616a] - Wandb v0.5.1
  [f8abcde7] - micromamba_jll v1.4.9+0
  [7b1f6079] - FileWatching
   Resolving package versions...
    Updating `~/.julia/environments/v1.6/Project.toml`
  [ad70616a] + Wandb v0.5.1
    Updating `~/.julia/environments/v1.6/Manifest.toml`
  [992eb4ea] + CondaPkg v0.2.22
  [0f8b85d8] + JSON3 v1.14.0
  [0b3b1443] + MicroMamba v0.1.14
  [fa939f87] + Pidfile v1.3.0
  [6099a3de] + PythonCall v0.9.19
  [856f2bd8] + StructTypes v1.10.0
  [e17b2a0c] + UnsafePointers v1.0.0
  [ad70616a] + Wandb v0.5.1
  [f8abcde7] + micromamba_jll v1.4.9+0
  [7b1f6079] + FileWatching
Precompiling project...
  ✓ FreeType


## Import experimental data

In [2]:
data_exp = DataFrame(CSV.File("/Users/nmt121/Desktop/MODEL FINAL/S5_short_hyst.csv"))
hysteresis_exp= (data_exp[:,2]./ data_exp[:,1]).-1; #force(derecruitment) - force(recruitment)

In [5]:
 hysteresis_exp

10-element Vector{Float64}:
  4.85638175634269
  0.4721306188585599
  0.1889146394196899
 -0.07020177327746557
  0.15811437111521287
  0.049547887333827534
 -0.0003751370558326972
 -0.21201250700645224
  0.03434495130876836
 -0.0443895274044207

## Initiliase parameters

In [10]:
######################  S-TYPE ##########################
#########################################################
nr_s = 10

# Radius and soma lengths
rheobase_S = LinRange(3.5,6.5,nr_s) .+ rand(Normal(0.0,0.05),nr_s)  
rs_S = LinRange(38.75,41.25,nr_s)./10000 
ls_S = LinRange(77.5,82.5,nr_s)./10000 
rms_S = LinRange(1.15, 1.05,nr_s).*1000
rd_S = LinRange(20.75,31.25,nr_s)./10000 
ld_S = LinRange(5500.,6800.,nr_s)./10000 
rmd_S = LinRange(14.4,10.70,nr_s).*1000 

#  Maximun conductances for each of the ION channels mS/cm^2 
max_gna = 30.0 
max_gkf_S = ones(nr_s)*4 
max_gks_S =LinRange(16,25,nr_s) 
max_gca_S = LinRange(0.038,0.029,nr_s) 

# Rate constants of the kinetic processes in ms^-1
αm = 22.0; βm = 13.0
αh_S = ones(nr_s)*0.5
βh_S = ones(nr_s)*4
αn_S = ones(nr_s)*1.5
βn_S = ones(nr_s)*0.1
αq_S = ones(nr_s)*1.5
βq_S = ones(nr_s)*0.025
αp = 0.008
βp= LinRange(0.014,0.016,nr_s)
βp_S = ones(nr_s)*0.014

Vthca_S = LinRange(2.5,3.0,nr_s) 
Apeak_S = LinRange(0.103,0.123,nr_s)
tpeak_S = LinRange(110.,100.,nr_s) 
tetanic_S = LinRange(0.393,0.491,nr_s);

#EXCITATORY PROFILE
amplitude=0.4
tr_length=28000
half_triang = div(tr_length, 2)  
flat_line = fill(-0.1, 1000)
up = range(-0.1, amplitude, length = half_triang + 1)  
down = range(amplitude, -0.1, length = tr_length - half_triang + 1)  
tr = vcat(flat_line, up[2:end], down[2:end], flat_line)
sim_length=30000


γh= zeros(nr_s)

IF= LinRange(1.,1.,nr_s);  
ri = 70.;
cm = 1.; 
MN_list=[];


## Creation of the dictionary

Vth_S = zeros(nr_s); gls_S = zeros(nr_s); gld_S = zeros(nr_s); gc_S = zeros(nr_s); gna_S = zeros(nr_s); gkf_S = zeros(nr_s); gks_S = zeros(nr_s); gca_S = zeros(nr_s); rn_S = zeros(nr_s); cs_S = zeros(nr_s)  ; cd_S = zeros(nr_s)
for i in 1:nr_s 
    gls_S[i] = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6  
    gld_S[i] = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6 
    gc_S[i] = ((2/(((ri*ld_S[i])/(π*rd_S[i]^2))+((ri*ls_S[i])/(π*rs_S[i]^2)))))*1e6    
    rn_S[i] = 1/(gls_S[i]+((gld_S[i]*gc_S[i])/(gld_S[i]+gc_S[i])))    
    Vth_S[i] = round(rn_S[i]*rheobase_S[i],digits=2)   
    cs_S[i] = 2*π*ls_S[i]*rs_S[i]*cm*1e3  
    cd_S[i] = 2*π*ld_S[i]*rd_S[i]*cm*1e3   
    gna_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3          
    gkf_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3     
    gks_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3     
    gca_S[i] = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3     

    S_MN = Dict(:Cs => cs_S[i], :Cd => cd_S[i], :Irh => rheobase_S[i], :gc => gc_S[i], :gNa => gna_S[i], :gKs => gks_S[i], :gKf => gkf_S[i], :gCa => gca_S[i], :gld => gld_S[i], :gls => gls_S[i], :αp => αp, :βp => βp_S[i], 
        :αq => αq_S[i], :βq => βq_S[i], :αm => αm, :βm => βm, :αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i],:αp => αp, :βp => βp[i],  :Vth => Vth_S[i], :VthCa => Vthca_S[i], :Rn => rn_S[i],
        :Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],:IF => IF[i], :tpeak => tpeak_S[i], :γh=>γh[i] , "type" => "S Type")
    
    MN_list = [MN_list;S_MN];

end
MNs = MN_list;

EL=0; ENa=120; ECa=140; EK=-10; Ee=70; Ei=-16 
Ge=0.6; Gi=0.5 ; 
Vd0 = 0.; 
Vs0 = 0.;
rp=5.0; 
step = 0.05;
_int(num) = floor(Int, num);
pd=0.6; 

tspan = 0:0.05:sim_length; 
m=0.0;h=1.0;n=0.0;q=0.0;p=0.0;m0=0.0;h0=1.0;n0=0.0;q0=0.0;p0=0.0; t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s) 
sim_time = _int(sim_length/0.05)  
spike_arr = zeros((nr_s, sim_time)) 
vs_arr = zeros((nr_s, sim_time)); 
commoninput = tr; 
full_input = zeros((nr_s, sim_length))
independantnoise = zeros((nr_s, sim_length))
factor=0.7  
inh_input=fill(factor*amplitude, sim_length);


In [5]:

# Plots.plot( rs_S[2:20], hysteresis_exp[2:20], grid=false,legend=false, xlabel="Radius S",ylabel="Force at recruitment")



In [15]:
γh_initial= γh.+0.5

10-element Vector{Float64}:
 0.5
 0.5
 0.5
 0.5
 0.5
 0.5
 0.5
 0.5
 0.5
 0.5

In [7]:
# x_values = LinRange(-1, 1, nr_s)
# A = 1.5  # amplitude of the exp
# k = 10.0  # control rate ramp up and ramp down
# γh = zeros(nr_s)
# for i in 1:nr_s
#     x = x_values[i]
#     if x < 0
#         γh[i] = A * exp(k * x)
#     else
#         γh[i] = A * exp(-k * x)
#     end
# end
# γh_initial = γh .+ 0.05

In [11]:
#############################################################
#############################################################
############ FUNCTIONS FOR OPTIMISATION #####################
#############################################################
#############################################################


function object_function_constant_inh(γh,i)
    simulated_hysteresis, _ = simulate_hh_hysteresis_constant_inh(γh,i,MN_list)
    ssd = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
    return ssd 
end



function run_sim_optim_constant_inh(i)

    full_input = commoninput 
    _, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input, inh_input, MNs)       
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end


function firing_rates_max(spike_array,i)

    max_rates_mns=[]
    for i in 1: size(spike_array)[1]
        spike_pos=[]
        for j in 1:size(spike_array)[2]
            if spike_array[i,j]==1.0
                push!(spike_pos,j)
            end
        end
        ISI= diff(spike_pos);
        rates= 1 ./(ISI/20000)
        max_rates= maximum(rates)
        max_rates_mns=max_rates
        
    end
    
    max_rate_mn= max_rates_mns;
    
    return max_rate_mn;
    
    end


function simulate_hh_hysteresis_constant_inh(γh::Vector,i,MN_list)    
    MN_list[i][:γh] = γh[i]

    MNs= MN_list[i]

    spike_array, _ = run_sim_optim_constant_inh(i);

    amplitude=0.4;
    sim_length=560000 ;
    half_triang = div(sim_length, 2);  
    flat_line = fill(-0.1, 20000);
    up = range(-0.1, amplitude, length = half_triang + 1)  ;
    down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
    tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
    equiv_force_profile= (tr/maximum(tr))*nr_s

    hysteresis= []

    if any(spike_array[1, :] .== 1)
            
        value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
        value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold
        hysteresis= (value_2/value_1).-1

    else
        hysteresis=500 #random high value
    end

    max_firing_rates= firing_rates_max(spike_array,i)
    
    return hysteresis, max_firing_rates
end


function compute_gradient_manual!(g::Vector{Float64}, γh::Vector{Float64}, i::Int; ε::Float64=1e-3)
    γh_plus = copy(γh)
    γh_minus = copy(γh)
    
    γh_plus[i] += ε
    γh_minus[i] -= ε

    f_plus = BigFloat(object_function_constant_inh(γh_plus, i))
    f_minus = BigFloat(object_function_constant_inh(γh_minus, i))

    g[i] = (f_plus - f_minus) / (2 * ε)
    

end

function gradient_descent_optimization_constant_inh(γh_initial::Vector{Float64}, i::Int, learning_rate::Float64, max_iterations::Int, tolerance::Float64)
    
    γh = copy(γh_initial) #make a copy of the initial values of the hysteresis profile
    g = zeros(Float64, length(γh))  # Gradient vector initialization
    obj_value_prev = BigFloat(object_function_constant_inh(γh, i))  # Initial objective function value

    for iter in 1:max_iterations #for loop for going through all the iterations for ONE motoneuron
        fill!(g, 0.0)  # Reset gradient vector
        compute_gradient_manual!(g, γh, i)  # Compute gradient
        
        for j in 1:nr_s
            γh[j] -= learning_rate * g[j] #new value fo the hysteresis profile, doing it for all MNs, but in reality later only taking the one we are interested in 
        end

        γh = clamp.(γh, 0.0, 1.0)  # Ensure γh remains within bounds [0, 1]

        obj_value_new = BigFloat(object_function_constant_inh(γh, i)) # Compute new objective function value for the motoneuron we want

        if abs(obj_value_new - obj_value_prev) < tolerance #
            println("Convergence achieved at iteration $iter")
            break  
        end
        

        obj_value_prev = obj_value_new  # Update previous objective function value for next iteration

        println("Iteration $iter, Objective Function Value: ", obj_value_new)
    end

    return γh  
end






gradient_descent_optimization_constant_inh (generic function with 1 method)

## Optimisation

In [17]:
function object_function(γh,i)
    penalty = 0
    penalty_factor = 1e6  # Adjust

    if γh[i] < 0. || γh[i] > 1.
        penalty += penalty_factor * (0 - γh[i])^2
    end

    simulated_hysteresis, max_firing_rates = simulate_hh_hysteresis(γh,i)
    ssd = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
    
    return ssd + penalty
end


function run_sim_optim(i)

    full_input = commoninput 
    vd, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input, full_input*factor, MNs)       
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end


function firing_rates_max(spike_array,i)

        max_rates_mns=[]
    for i in 1: size(spike_array)[1]
        spike_pos=[]
        for j in 1:size(spike_array)[2]
            if spike_array[i,j]==1.0
                push!(spike_pos,j)
            end
        end
        ISI= diff(spike_pos);
        rates= 1 ./(ISI/20000)
        max_rates= maximum(rates)
        max_rates_mns=max_rates
        
    end
    
    max_rate_mn= max_rates_mns;
    
    return max_rate_mn;
    
    end


function simulate_hh_hysteresis(γh::Vector,i)    
    MN_list[i][:γh] = γh[i]

    MNs= MN_list[i]

    spike_array, vs_array = run_sim_optim(i);

    amplitude=0.4;
    sim_length=560000 ;
    half_triang = div(sim_length, 2);  
    flat_line = fill(-0.1, 20000);
    up = range(-0.1, amplitude, length = half_triang + 1)  ;
    down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
    tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
    equiv_force_profile= (tr/maximum(tr))*nr_s

    hysteresis= []

    value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
    value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold

    if value_1 !== nothing
        # Safe to use index
    else
        print(11111111)
    end

    if value_2 !== nothing
        # Safe to use index
    else
        print(222222222)
    end
    hysteresis= (value_2/value_1).-1
    # println(hysteresis)

    max_firing_rates= firing_rates_max(spike_array,i)
    
    return hysteresis, max_firing_rates
end


function compute_gradient_manual!(g::Vector{Float64}, γh::Vector{Float64}, i::Int; ε::Float64=1e-3)
    γh_plus = copy(γh)
    γh_minus = copy(γh)
    
    γh_plus[i] += ε
    γh_minus[i] -= ε

    f_plus = BigFloat(object_function(γh_plus, i))
    f_minus = BigFloat(object_function(γh_minus, i))

    g[i] = (f_plus - f_minus) / (2 * ε)
end

function gradient_descent_optimization(γh_initial::Vector{Float64}, i::Int, learning_rate::Float64, max_iterations::Int, tolerance::Float64)
    
    γh = copy(γh_initial)  # Initial parameters
    g = zeros(Float64, length(γh))  # Gradient vector initialization
    obj_value_prev = BigFloat(object_function(γh, i))  # Initial objective function value

    for iter in 1:max_iterations
        fill!(g, 0.0)  # Reset gradient vector
        compute_gradient_manual!(g, γh, i)  # Compute gradient

        
        for j in 1:length(γh)
            γh[j] -= learning_rate * g[j]
        end

        # Ensure γh remains within bounds [0, 1]
        γh = clamp.(γh, 0.0, 1.0)

        # Compute new objective function value
        obj_value_new = BigFloat(object_function(γh, i))


        if abs(obj_value_new - obj_value_prev) < tolerance
            println("Convergence achieved at iteration $iter")
            break  
        end

        obj_value_prev = obj_value_new  # Update previous objective function value for next iteration

        println("Iteration $iter, Objective Function Value: ", obj_value_new)
    end

    return γh  
end



gradient_descent_optimization (generic function with 2 methods)

In [12]:
optimized_γh=zeros(1,nr_s)
for i in 1:nr_s
    println(i)
    if hysteresis_exp[i]<0
        optimized_γh[1,i]= gradient_descent_optimization_constant_inh(γh_initial,i, 0.001, 100,1e-6)[i]
        println(optimized_γh[1,i])
    else
        optimized_γh[1,i]= γh_initial[i]
        println("No Hysteresis")

    end
end

1
No Hysteresis
2
No Hysteresis
3
No Hysteresis
4
Convergence achieved at iteration 1
0.0
5
No Hysteresis
6
No Hysteresis
7
Convergence achieved at iteration 1
0.0
8


ArgumentError: ArgumentError: reducing over an empty collection is not allowed

In [60]:
optimized_γh[1:10]

10-element Vector{Float64}:
 0.05006809989464373
 0.05062841318517258
 0.055798880209209215
 0.10351099002087855
 0.5433042529655824
 0.5444886490103167
 0.13541465860474394
 0.055798880209209215
 0.061345798467742504
 0.05006809989464373

In [8]:
################## OPTIMISATION USING LIBRARIES ##################
################################################################## 

global current_γh = copy(γh_initial)  

function objective_function(γh::Vector)
    total_error = 0.0
    for i in 1:length(γh)
        simulated_hysteresis, _ = simulate_hh_hysteresis_constant_inh(γh, i, MN_list)
        error_i = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
        total_error += error_i
        println("Current γh: ", γh[i], " | Total Error: ", total_error)
    end
    
    return total_error
end


function objective_wrapper(γh_single_value::Vector{Float64}, i::Int)
    temp_γh = copy(current_γh)
    temp_γh[i] = γh_single_value[1]
    
    return objective_function(temp_γh)
end




for i in 1
    γh_init = [current_γh[i]]  # Initial value for the i-th γh

    opt_result = optimize(x -> objective_wrapper(x, i), γh_init, BFGS(), Optim.Options(g_tol = 1e-6))

    current_γh[i] = clamp(Optim.minimizer(opt_result)[1], 0.0, 1.0)

end

println("Optimized γh values for each motor neuron: ", current_γh)




Current γh: 0.05007415534909612 | Total Error: 23.052405779296596
Current γh: 0.05062841318517258 | Total Error: 23.201876901708534
Current γh: 0.055798880209209215 | Total Error: 23.214806188382287
Current γh: 0.10351099002087855 | Total Error: 23.21634584595144
Current γh: 0.5437894817118581 | Total Error: 23.308262736740662
Current γh: 0.5437894817118581 | Total Error: 23.34569974711596
Current γh: 0.10351099002087863 | Total Error: 23.34583110526303
Current γh: 0.055798880209209215 | Total Error: 23.40904026618236
Current γh: 0.05062841318517258 | Total Error: 23.4092933876501
Current γh: 0.05006809989464373 | Total Error: 23.414492773932803
Current γh: 0.05006204444019134 | Total Error: 23.069065075722815
Current γh: 0.05062841318517258 | Total Error: 23.218536198134753
Current γh: 0.055798880209209215 | Total Error: 23.231465484808506
Current γh: 0.10351099002087855 | Total Error: 23.23300514237766
Current γh: 0.5437894817118581 | Total Error: 23.32492203316688
Current γh: 0.5437

InterruptException: InterruptException:

In [16]:
using Optim

# Assuming `current_γh` is your initial parameters vector
global current_γh = γh_initial # Example initial values

# Define your objective function here
function objective_function(γh::Vector)
    total_error = 0.0
    for i in 1:length(γh)
        simulated_hysteresis, _ = simulate_hh_hysteresis_constant_inh(γh, i, MN_list)
        error_i = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
        total_error += error_i
        println("Current γh: ", γh[i], " | Total Error: ", total_error)
    end
    
    return total_error
end
# Optimizes a single parameter within γh, given its index
function optimize_single_parameter(index)
    global current_γh
    
    # Objective wrapper for a single parameter optimization
    function objective_wrapper(x::Vector{Float64})
        modified_γh = copy(current_γh)
        modified_γh[index] = x[1]
        return objective_function(modified_γh)
    end

    # Setup bounds for the single parameter being optimized
    lb = [0.0]
    ub = [1.0]
    initial_guess = [current_γh[index]]

    # Perform optimization
    opt_result = optimize(objective_wrapper, lb, ub, initial_guess, Fminbox(BFGS()), Optim.Options(show_trace=true))

    # Update the optimized parameter back into global current_γh
    current_γh[index] = Optim.minimizer(opt_result)[1]
end

# Example: Optimize the 4th parameter
optimize_single_parameter(4)

println("Optimized γh values: ", current_γh)


Current γh: 0.5 | Total Error: 25.048406154713472
Current γh: 0.5 | Total Error: 25.44134876209732
Current γh: 0.5 | Total Error: 25.55404427482391
Current γh: 0.5000060554544524 | Total Error: 25.55855236431746
Current γh: 0.5 | Total Error: 25.643050362571902
Current γh: 0.5 | Total Error: 25.674340483474275
Current γh: 0.5 | Total Error: 25.69348337966649
Current γh: 0.5 | Total Error: 25.69908071207271
Current γh: 0.5 | Total Error: 25.732106401745103
Current γh: 0.5 | Total Error: 25.740628025867693
Current γh: 0.5 | Total Error: 25.048406154713472
Current γh: 0.5 | Total Error: 25.44134876209732
Current γh: 0.5 | Total Error: 25.55404427482391
Current γh: 0.49999394454554763 | Total Error: 25.55855236431746
Current γh: 0.5 | Total Error: 25.643050362571902
Current γh: 0.5 | Total Error: 25.674340483474275
Current γh: 0.5 | Total Error: 25.69348337966649
Current γh: 0.5 | Total Error: 25.69908071207271
Current γh: 0.5 | Total Error: 25.732106401745103
Current γh: 0.5 | Total Error